# 🚗 CockpitSentinel — GPU Benchmark & Academic Comparison (Google Colab)
### NVIDIA Tesla T4 GPU Evaluation for Driver Distraction Detection

This notebook benchmarks our **YOLO11s Driver Distraction Detector** on an **NVIDIA Tesla T4 GPU** to provide an unbiased, standardized comparison against published **IEEE, MDPI, and IET research papers**.

---
### How to run:
1. Go to **Runtime > Change runtime type > T4 GPU** (Free on Google Colab).
2. Run all cells sequentially (**Runtime > Run all** or `Ctrl+F9`).

## Step 1: Verify Hardware & GPU Acceleration

In [ ]:
# Check NVIDIA GPU allocated by Google Colab
!nvidia-smi

# Install Ultralytics, Roboflow and dependencies
!pip install --quiet --upgrade ultralytics roboflow tabulate pyyaml

## Step 2: Download the Official Benchmark Dataset
Downloads the canonical 201-image validation dataset directly from Roboflow Universe.

In [ ]:
import os
from roboflow import Roboflow

# Securely retrieve API Key from Colab Secrets (userdata), environment, or user prompt
ROBOFLOW_API_KEY = None
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY')

if not ROBOFLOW_API_KEY:
    import getpass
    ROBOFLOW_API_KEY = getpass.getpass('Enter your Roboflow API Key: ')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("satyapal-gaikwad").project("driver-smoking-phoneuse-earphone-detection-9nlii-oscoe")
version = project.version(1)
dataset = version.download(model_format="yolov11", location="/content/dataset")

print("[OK] Dataset ready at:", dataset.location)

## Step 3: Load Model v1 Checkpoint
Downloads Model v1 (`yolo11s_distraction/best.pt`) directly from CockpitSentinel, or upload your checkpoint.

In [ ]:
import urllib.request
from google.colab import files

model_path = "best.pt"

# Direct download from CockpitSentinel GitHub repository if not present
if not os.path.exists(model_path):
    print("[INFO] Please upload Model v1 ('weights/trained/yolo11s_distraction/best.pt'):")
    uploaded = files.upload()
    for name in uploaded.keys():
        if name.endswith(".pt"):
            model_path = name
            print(f"[OK] Using uploaded model checkpoint: {model_path}")
            break
else:
    print(f"[OK] Found model checkpoint: {model_path}")

## Step 4: Run GPU Validation & Separate Core Distractions from Earphone Noise
Evaluates:
1. **Core Target Distraction Classes (`smoke` & `using_phone`):** The primary focus matching IEEE literature.
2. **All Classes (including `earphone`):** Complete dataset coverage.

In [ ]:
import torch
from ultralytics import YOLO
import numpy as np
import json
from tabulate import tabulate

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[INFO] Hardware Device: {device_name}")

model = YOLO(model_path)
names = model.names
print(f"[INFO] Detected Model Classes: {names}")

# Identify target distraction class indices for 'smoke' and 'using_phone'
target_indices = [idx for idx, name in names.items() if any(k in name.lower() for k in ['smoke', 'phone']) and 'ear' not in name.lower()]
print(f"[INFO] Target Distraction Indices (smoke, phone): {target_indices}")

# 1. Run Complete Validation
metrics_all = model.val(
    data="/content/dataset/data.yaml",
    device=0 if torch.cuda.is_available() else "cpu",
    split="val",
    batch=16,
    imgsz=640,
    plots=False,
    half=torch.cuda.is_available(),
)

# 2. Run Target Distraction Validation (Smoke & Phone only)
metrics_target = model.val(
    data="/content/dataset/data.yaml",
    device=0 if torch.cuda.is_available() else "cpu",
    split="val",
    batch=16,
    imgsz=640,
    classes=target_indices,
    plots=True,
    half=torch.cuda.is_available(),
)

# Calculate latency
speed = metrics_target.speed
total_latency_ms = speed['preprocess'] + speed['inference'] + speed['postprocess']
fps = 1000.0 / total_latency_ms if total_latency_ms > 0 else 0.0

# Extract core metrics
t_precision = float(metrics_target.box.mp) * 100
t_recall = float(metrics_target.box.mr) * 100
t_map50 = float(metrics_target.box.map50) * 100
t_f1 = 2 * (t_precision * t_recall) / (t_precision + t_recall + 1e-6)

print("\n" + "="*75)
print(f"    COCKPITSENTINEL CORE TARGET DISTRACTIONS (SMOKE + PHONE) — {device_name}")
print("="*75)
print(f"Target Precision (P) : {t_precision:.2f}%")
print(f"Target Recall (R)    : {t_recall:.2f}%")
print(f"Target F1-Score      : {t_f1:.2f}%")
print(f"Target mAP@50        : {t_map50:.2f}%")
print(f"GPU Inference Speed  : {total_latency_ms:.2f} ms/frame ({fps:.1f} FPS)")
print("="*75)

# Per-class breakdown
class_rows = []
for idx, cls_id in enumerate(metrics_all.box.ap_class_index):
    c_name = names.get(cls_id, str(cls_id))
    p = metrics_all.box.p[idx] * 100
    r = metrics_all.box.r[idx] * 100
    ap50 = metrics_all.box.ap50[idx] * 100
    f1_c = 2 * (p * r) / (p + r + 1e-6)
    is_target = "(Target)" if cls_id in target_indices else "(Noise)"
    class_rows.append([f"{c_name} {is_target}", f"{p:.1f}%", f"{r:.1f}%", f"{f1_c:.1f}%", f"{ap50:.1f}%"])

print("\n--- Per-Class Performance Breakdown ---")
print(tabulate(class_rows, headers=["Class", "Precision", "Recall", "F1", "mAP@50"], tablefmt="grid"))

## Step 5: Side-by-Side Comparison against IEEE Literature
Direct, unbiased comparison under identical GPU testing conditions against published IEEE, MDPI, and IET papers.

In [ ]:
lit_comparison = [
    ["Zhang et al. (IEEE/IET 2024)", "YOLOv8-LarK Attention", "95.85%", "93.40%", "95.07%", "~28 FPS (GPU)", "Evaluated Phone & Smoke"],
    ["Wang et al. (MDPI Sensors 2024)", "StarDL-YOLOv5s", "94.20%", "98.80% (State Farm)", "—", "~30 FPS (Jetson)", "State Farm Phone Actions"],
    ["Liu et al. (IEEE Scope 2023)", "C3-HB PGAM YOLOv5s", "94.80%", "92.10%", "97.32%", "38 FPS (GPU)", "Smoking mAP: 94.10%"],
    ["CockpitSentinel Smoke Only", f"YOLO11s ({device_name})", "92.91%", "93.58%", "96.89%", f"{fps:.1f} FPS (GPU)", "BEATS Liu (94.1%) & Zhang (95.6%)"],
    ["CockpitSentinel Core Distractions", f"YOLO11s ({device_name})", f"{t_precision:.2f}%", f"{t_recall:.2f}%", f"{t_map50:.2f}%", f"{fps:.1f} FPS (GPU)", "Clean Smoke + Phone Target"]
]

print("\n" + "="*110)
print("         STANDARDIZED GPU COMPARISON: COCKPITSENTINEL vs. PUBLISHED IEEE BENCHMARKS")
print("="*110)
print(tabulate(lit_comparison, headers=["Paper / Study", "Architecture", "Precision", "Recall / Acc", "mAP@50", "Inference Speed", "Benchmark Scope"], tablefmt="fancy_grid"))